In [1]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the data
trends = pd.read_csv('ai_tools_comparison.csv')

# Inspect the data
trends.head()

FileNotFoundError: [Errno 2] No such file or directory: 'ai_tools_comparison.csv'

In [ ]:
# Data overview
print(trends.info())
print(trends.columns)
print(trends.head())

In [ ]:
# Rename columns for easier access
trends.columns = ['week', 'chatgpt', 'gemini', 'microsoft_copilot']

# Convert week to datetime
trends['week'] = pd.to_datetime(trends['week'])

# Check for missing values
print(trends.isnull().sum())
print(trends.describe())

In [ ]:
# 1. Which AI tool has shown the most consistent growth over the observed period?
# Calculate linear trend using slope for each tool

from scipy.stats import linregress

def calculate_slope(series):
    x = range(len(series))
    slope, _, _, _, _ = linregress(x, series)
    return slope

slopes = {}
for tool in ['chatgpt', 'gemini', 'microsoft_copilot']:
    slopes[tool] = calculate_slope(trends[tool])

most_consistent_tool = max(slopes, key=slopes.get)
print(f"Most consistent growth tool: {most_consistent_tool}")

In [ ]:
# 2. Visualization of ChatGPT, Gemini, and Microsoft Copilot over time
plt.figure(figsize=(12, 6))
plt.plot(trends['week'], trends['chatgpt'], label='ChatGPT', linewidth=2)
plt.plot(trends['week'], trends['gemini'], label='Gemini', linewidth=2)
plt.plot(trends['week'], trends['microsoft_copilot'], label='Microsoft Copilot', linewidth=2)
plt.xlabel('Date')
plt.ylabel('Interest')
plt.title('Interest Over Time: ChatGPT vs Gemini vs Microsoft Copilot')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Find ChatGPT's largest decline month
# Calculate month-over-month change for ChatGPT
trends['month'] = trends['week'].dt.to_period('M')
monthly_avg = trends.groupby('month')['chatgpt'].mean().reset_index()

# Calculate percentage change between consecutive months
monthly_avg['pct_change'] = monthly_avg['chatgpt'].pct_change() * 100

# Find the largest decline (most negative percentage change)
largest_decline_idx = monthly_avg['pct_change'].idxmin()
gpt_dip_month = monthly_avg.loc[largest_decline_idx, 'month']

# Format as "Month YYYY"
gpt_dip = gpt_dip_month.strftime('%B %Y')
print(f"ChatGPT's largest decline: {gpt_dip}")

In [ ]:
# 3. Seasonality: monthly averages across all tools
# Melt the dataframe for easier grouping
trends_melted = trends.melt(id_vars=['week', 'month'],
                            value_vars=['chatgpt', 'gemini', 'microsoft_copilot'],
                            var_name='tool',
                            value_name='interest')

# Calculate monthly average across all tools
monthly_all_avg = trends_melted.groupby('month')['interest'].mean().reset_index()

# Find month with highest average interest
best_month_row = monthly_all_avg.loc[monthly_all_avg['interest'].idxmax()]
best_month = best_month_row['month'].strftime('%B')
print(f"Month with highest average interest across all tools: {best_month}")

In [ ]:
# Final outputs
print("\n=== Final Answers ===")
print(f"most_consistent_tool: {most_consistent_tool}")
print(f"gpt_dip: {gpt_dip}")
print(f"best_month: {best_month}")